# Urdu Orpheus stage 3: build the v4 dataset (UAT + ASLP + UrduTTS)

**One run, no decisions.** Before you start: enable **GPU (T4)** and **Internet**, attach the `HF_TOKEN` secret
(Add-ons > Secrets, a Hugging Face token with write access to the owner's checkpoint repo), then
**Save Version > Save & Run All**. Takes about 2.5-3 hours of GPU. Nothing else is needed.

What it does:
1. A smoke test of the whole pipeline on 48 clips (about 10 minutes). If this fails, nothing was published.
2. Downloads the UAT corpus, converts it to 24 kHz mono, measures audio quality (DNSMOS/VAD).
3. Checks every UAT transcript against its audio with MMS CTC alignment and drops the worst-matching clips.
4. Re-selects the ASLP conversational and UrduTTS clips, builds a ~40 h train / validation split with voice labels.
5. Verifies every file, uploads the private dataset, and publishes the pointer the training notebook reads.
6. Loads the published dataset with the trainer's own loader as a final consistency check.

It succeeded if the last cell prints `STAGE-3 DATA READY`. Then run the training notebook.
UAT has no declared licence: the dataset it creates is private and for this course project only.


In [ ]:
import os, sys, subprocess, re, time
NOTEBOOK_START = time.time()
from pathlib import Path

REPO_URL = "https://github.com/hassan-31x/orpheus-ft-aslp-urdutts.git"
REPO_REF = "PINNED_BY_PUSH_SCRIPT"  # Pinned commit; keep identical for every epoch and every account.
PROJECT_FOLDER = "."  # This standalone GitHub repository contains the scripts at its root.
HF_REPO_ID = "hassan-31x/orpheus-urdu-stage2-checkpoints"  # Private Hub MODEL repo holding every stage-2 epoch run.
HF_SECRET_NAME = "HF_TOKEN"  # Exact Kaggle secret LABEL used in your previous notebook; change if different.
SMOKE_ONLY = False  # True: run only the 10-minute smoke test (debugging); the full build is skipped.
LEARNING_RATE = None  # None uses the per-epoch value in stage2_plan.py. Keep fixed within one epoch.
TEST = True  # True: few-clip smoke run of this epoch. Set False after "TEST PASSED" for the full epoch.
TEST_TRAIN_CLIPS = 24
TEST_VALIDATION_CLIPS = 8
AUDIT_ONLY = False
QUALITY_DEVICE = "cuda"
SESSION_HOURS = 11.75  # Kaggle kills sessions at 12 h; training saves + uploads a checkpoint before that.
CLONE = Path("/kaggle/working/orpheus-mixed-code")

if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", HF_REPO_ID):
    raise ValueError("HF_REPO_ID must be username/repository")
def load_hf_token(secret_name="HF_TOKEN", environ=None, secret_client_factory=None):
    # Keep a valid explicit environment token; do not overwrite it with a failed Kaggle lookup.
    environ = os.environ if environ is None else environ
    for name in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        value = environ.get(name, "").strip()
        if value:
            environ["HF_TOKEN"] = value
            return "environment:" + name
    for _path in sorted(Path("/kaggle/input").glob("**/hf_token.txt")):
        value = _path.read_text().strip()
        if value:
            environ["HF_TOKEN"] = value
            return "attached private Kaggle dataset"
    if not isinstance(secret_name, str) or not secret_name.strip():
        raise RuntimeError("HF_SECRET_NAME must be the exact label in Kaggle Add-ons > Secrets")
    try:
        if secret_client_factory is None:
            from kaggle_secrets import UserSecretsClient
            secret_client_factory = UserSecretsClient
        import time as _time
        for _attempt in range(3):  # the secrets service can refuse connections in API-started sessions
            try:
                value = secret_client_factory().get_secret(secret_name)
                break
            except ConnectionError:
                if _attempt == 2:
                    raise
                _time.sleep(10)
    except Exception as exc:
        # Never display exception text: upstream responses can contain credentials.
        error_type = type(exc).__name__
        raise RuntimeError(
            "Kaggle could not read the Hugging Face secret (" + error_type + "). "
            "Open this notebook in Edit mode > Add-ons > Secrets. Add/select the secret "
            "whose label exactly matches HF_SECRET_NAME and ENABLE its access for THIS notebook. "
            ""
            "Use a Hugging Face token with write access to " + HF_REPO_ID + ". "
            "Then Save Version > Save & Run All again. Do not paste the token into notebook code."
        ) from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError("The selected Kaggle secret is empty; update its value in Add-ons > Secrets")
    environ["HF_TOKEN"] = value.strip()
    return "Kaggle Secrets"

credential_source = load_hf_token(HF_SECRET_NAME)
print("Hugging Face credential loaded from", credential_source, "(value hidden)")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
if not AUDIT_ONLY or QUALITY_DEVICE == "cuda":
    try:
        gpu_list = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        gpu_list = ""
    if "GPU" not in gpu_list:
        raise RuntimeError("Enable an NVIDIA GPU in Kaggle Settings; check remaining GPU quota")
    print(gpu_list.strip())
if not CLONE.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CLONE)], check=True)
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=CLONE, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=CLONE, check=True)
REPO = CLONE / PROJECT_FOLDER if (CLONE / PROJECT_FOLDER).is_dir() else CLONE
for name in ("setup_kaggle.py", "finetune_mixed.py", "stage2_data.py", "stage2_plan.py", "checkpoint_init.py", "urdutts_data.py", "audio_storage.py", "workflow_gate.py", "configs/mixed.json", "configs/stage2.json", "stage3_build.py", "build_stage2.py", "stage2_labels.py"):
    if not (REPO / name).is_file():
        raise RuntimeError(f"Missing {name}: push the mixed-data folder and check REPO_REF/PROJECT_FOLDER")
PROJECT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CLONE, text=True).strip()
print("Project commit (pin REPO_REF to this for resumes):", PROJECT_COMMIT)
early_check = subprocess.run([sys.executable, str(REPO / "check_hub.py"), "--repo", HF_REPO_ID], cwd=REPO)
if early_check.returncode == 2:
    raise RuntimeError("Hugging Face storage check failed; see redacted diagnostics above")
if early_check.returncode:
    print("Early Hub check unavailable in host Python; it will run in the installed training venv")

Cell 2: install and preflight

In [ ]:
# Dedicated venv inherits Kaggle CUDA wheels; installs project dependencies every fresh session.
ENV = Path("/kaggle/working/orpheus-mixed-env")
TRAIN_PYTHON = str(ENV / "bin" / "python")
setup_command = [sys.executable, "-u", str(REPO / "setup_kaggle.py"), "--venv", str(ENV)]
if not AUDIT_ONLY:
    setup_command.append("--gpu")
subprocess.run(setup_command, cwd=REPO, check=True)
subprocess.run([TRAIN_PYTHON, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=REPO, check=True)
subprocess.run(["df", "-h", "/kaggle/working"], check=True)

Cell 3: smoke test of the pipeline

In [ ]:

import shutil, time


def tail(path, lines=40):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 200_000))
            return "\n".join(f.read().decode("utf-8", "replace").splitlines()[-lines:])
    except OSError:
        return ""


def run_logged(command, log, tail_every=300):
    """Run a command with its output in a FILE (a stalled Kaggle log stream can block a chatty process)."""
    os.environ["PYTHONFAULTHANDLER"] = "1"
    log = Path(log)
    log.parent.mkdir(parents=True, exist_ok=True)
    print("Output ->", log, flush=True)
    with log.open("ab") as console:
        process = subprocess.Popen(command, cwd=REPO, stdout=console, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)
        last = time.time()
        while process.poll() is None:
            time.sleep(15)
            if time.time() - last >= tail_every:
                print(f"--- {time.strftime('%H:%M:%S')} last lines ---", flush=True)
                print(tail(log, 6), flush=True)
                last = time.time()
    print(tail(log, 60), flush=True)
    if process.returncode:
        raise RuntimeError(f"{Path(command[2]).name} exited with {process.returncode}; the traceback is above")

# 1/2 Smoke test: the whole pipeline on 48 UAT clips and one part of each old pool. Nothing is uploaded.
SMOKE = Path("/kaggle/working/stage3-smoke")
run_logged([TRAIN_PYTHON, "-u", str(REPO / "stage3_build.py"), "--phase", "all", "--work", str(SMOKE),
            "--limit", "48", "--pool-parts", "1", "--no-hub", "--allow-out-of-range"], "/kaggle/working/stage3_smoke.log")
print("SMOKE TEST PASSED: ingest, scoring, alignment, selection and verification all worked.")
shutil.rmtree(SMOKE, ignore_errors=True)


Cell 4: build and publish the dataset

In [ ]:

# 2/2 The real build. Resumable: if the session dies, run the notebook again and it continues.
if SMOKE_ONLY:
    print("SMOKE_ONLY is set: stopping before the full build.")
else:
  run_logged([TRAIN_PYTHON, "-u", str(REPO / "stage3_build.py"), "--phase", "all", "--work", "/kaggle/working/stage3"],
             "/kaggle/working/stage3_full.log", tail_every=600)


Cell 5: consumer check

In [ ]:

def consumer_check():
    # Load the published dataset with the trainer's own code (first shard only) and print the summary.
    import json
    from huggingface_hub import hf_hub_download
    pointer = json.loads(Path(hf_hub_download(HF_REPO_ID, "datasets/stage3-v4.json")).read_text())
    print("Pointer:", pointer)
    report = json.loads(Path(hf_hub_download(pointer["dataset_id"], "report.json", repo_type="dataset",
                                             revision=pointer["dataset_revision"])).read_text())
    print("Train:", json.dumps(report["train"], indent=1, ensure_ascii=False))
    print("UAT screening:", json.dumps(report["uat"], indent=1, ensure_ascii=False))
    print("Conversational voice candidates:", json.dumps(report["conversational_voice_candidates"], ensure_ascii=False))
    check = f"""
import sys
sys.path.insert(0, {str(REPO)!r})
import stage2_plan, stage2_data
base = stage2_plan.load_base({str(REPO)!r})
cfg = stage2_plan.epoch_config(6, base, data_pointer=stage2_plan.load_data_pointer())
cfg.update(test_train_clips=24, test_validation_clips=8, work_dir="/kaggle/working/prep-check")
print("Loaded by the trainer:", stage2_data.prepare_dataset(cfg, cfg["work_dir"]))
"""
    subprocess.run([TRAIN_PYTHON, "-c", check], cwd=REPO, check=True)
    shutil.rmtree("/kaggle/working/prep-check", ignore_errors=True)
    print("STAGE-3 DATA READY")


if SMOKE_ONLY:
    print("SMOKE_ONLY: the smoke test passed and nothing was published.")
else:
    consumer_check()
